# Palace Differential GSGSG Simulation — Wave Ports

[Palace](https://awslabs.github.io/palace/) is an open-source 3D electromagnetic simulator supporting eigenmode, driven (S-parameter), and electrostatic simulations. This notebook is the differential counterpart to [palace_cpw_waveport.ipynb](./palace_cpw_waveport.ipynb): instead of a single-ended GSG coplanar waveguide it drives a **GSGSG** structure — two signal electrodes sharing a central ground — and extracts the **differential** and **common-mode** impedance and effective index.

The two signal electrodes sit on an 80 µm pitch, far enough apart that each behaves as its own CPW. That decoupling is what makes the port scheme below possible: rather than one wave port spanning the whole boundary face, each signal line gets **its own wave port** covering only its share of the width. Four ports (two per face) then give a full 4×4 single-ended S-matrix, which converts to mixed-mode parameters with `scikit-rf`.

A follow-up section loads the line with **T-bars** — the periodic capacitive loading an MZM modulator segment presents to a travelling-wave electrode — and shows how that lowers $Z_c$ and raises $n_{\mathrm{eff}}$.

**Requirements:**

- IHP PDK: `uv pip install ihp-gdsfactory`
- [GDSFactory+](https://gdsfactory.com) account for cloud simulation

### Define GSGSG electrode

In [ ]:
import gdsfactory as gf
from ihp import LAYER, PDK

PDK.activate()


@gf.cell
def gsgsg_electrode(
    length: float = 800,
    s_width: float = 20,
    g_width: float = 40,
    gap_width: float = 15,
    signal_pitch: float = 80,
    layer=LAYER.TopMetal2drawing,
) -> gf.Component:
    """
    Create a GSGSG (differential coplanar) electrode.

    The two signal electrodes sit on `signal_pitch` centres, separated by a
    shared central ground, so each line is its own CPW and the two are only
    weakly coupled.

    Args:
        length: horizontal length of the electrodes
        s_width: width of each signal electrode
        g_width: width of the two outer ground electrodes
        gap_width: gap between a signal electrode and its adjacent grounds
        signal_pitch: centre-to-centre spacing of the two signal electrodes
        layer: layer for the metal
    """
    c = gf.Component()

    center_g_width = signal_pitch - s_width - 2 * gap_width
    if center_g_width <= 0:
        raise ValueError(
            f"signal_pitch={signal_pitch} is too small for s_width={s_width} "
            f"and gap_width={gap_width}: the central ground would vanish."
        )

    s_center = signal_pitch / 2
    g_center = s_center + s_width / 2 + gap_width + g_width / 2

    # Central ground, shared by both lines
    c << gf.c.rectangle((length, center_g_width), centered=True, layer=layer)

    for sign in (+1, -1):
        sig = c << gf.c.rectangle((length, s_width), centered=True, layer=layer)
        sig.move((0, sign * s_center))

        gnd = c << gf.c.rectangle((length, g_width), centered=True, layer=layer)
        gnd.move((0, sign * g_center))

    # Port order matters: scikit-rf se2gmm(p=2) treats ports (0, 1) as the
    # left-hand pair and (2, 3) as the right-hand pair, with 0-2 and 1-3 the
    # two through paths. So: o1/o3 = upper line, o2/o4 = lower line.
    for name, x, orientation, sign in (
        ("o1", -length / 2, 180, +1),
        ("o2", -length / 2, 180, -1),
        ("o3", length / 2, 0, +1),
        ("o4", length / 2, 0, -1),
    ):
        c.add_port(
            name=name,
            center=(x, sign * s_center),
            width=s_width,
            orientation=orientation,
            port_type="electrical",
            layer=layer,
        )

    c.info["s_width"] = s_width
    c.info["g_width"] = g_width
    c.info["gap_width"] = gap_width
    c.info["signal_pitch"] = signal_pitch
    c.info["center_g_width"] = center_g_width
    return c


c = gsgsg_electrode()
cc = c.copy()
cc.draw_ports()
cc

### Configure simulation

Each signal line gets its own wave port, so the port rectangles must **not** span the full boundary face — `max_size=True` would make the two ports on a face identical and overlapping. Two settings control the extent instead:

- **`lateral_margin`** sets the half-width of the port box around the signal centre. At 25 µm the box spans $y \in [5, 75]$ for the upper line, so its edges land ~10 µm inside the central ground and ~10 µm inside the outer ground, where the fields are already small. Palace treats the port cross-section boundary as PEC in the port eigenproblem, and terminating it inside a ground conductor is a good approximation of that. The two boxes on a face are left 10 µm apart.
- **`full_height=True`** makes the port span the simulation domain in $z$. This matters: the air box is not part of the layer stack, so without it the port box is clamped to the stack and collapses onto the conductor itself, putting a PEC lid directly above the electrodes and badly corrupting $Z_c$ and $n_{\mathrm{eff}}$.

**All four ports are excited.** Palace assigns one excitation index per excited port, and the results parser fills missing S-matrix entries only by reciprocity — never by geometric symmetry. Exciting fewer ports would leave whole rows of the 4×4 at exactly zero, and the mixed-mode transform needs the complete matrix. Four excitations is ~4× the solve cost of the single-ended notebook, which is why the sweep is 1–50 GHz with 100 points.

In [ ]:
from gsim.common.stack import get_stack
from gsim.palace import DrivenSim

PORT_NAMES = ("o1", "o2", "o3", "o4")


def setup_sim(cell, output_dir="./palace-sim-gsgsg-waveport"):
    sim = DrivenSim()
    sim.set_output_dir(output_dir)
    sim.set_geometry(cell)

    stack = get_stack()  # auto-detects active PDK
    sim.set_stack(stack)
    sim.set_airbox(margin_x=0.0, margin_y=50, z_above=100.0, z_below=100.0)

    # One wave port per signal line per face. Partial width (so the two ports
    # on a face do not overlap) but full height in z.
    lateral_margin = cell.info["gap_width"] + cell.info["g_width"] / 4

    for name in PORT_NAMES:
        sim.add_wave_port(
            name,
            layer="topmetal2",
            lateral_margin=lateral_margin,
            full_height=True,
            mode=1,
            excited=True,
        )

    sim.set_driven(fmin=1e9, fmax=50e9, num_points=100)

    print(sim.validate_config())

    return sim


sim = setup_sim(c)

### Generate mesh

In [ ]:
sim.mesh(preset="default", refined_mesh_size=2.0, max_mesh_size=40.0, fmax=60e9)

In [ ]:
# Confirm the four port boxes are partial-width, full-height and non-overlapping
for p in sim._last_mesh_result.port_info:
    print(
        f"P{p['portnumber']}  x={p['xmin']:7.1f}  "
        f"y=[{p['ymin']:6.1f}, {p['ymax']:6.1f}]  "
        f"z=[{p['zmin']:7.1f}, {p['zmax']:7.1f}]"
    )

In [ ]:
sim.plot_mesh(
    style="solid",
    transparent_groups=["air__None", "SiO2__None", "SiO2__passive", "air__passive"],
    interactive=True,
)

### Run simulation

In [ ]:
results = sim.run(check_cache=True)

In [ ]:
results.plot_interactive()

## Differential and common-mode parameters

The 4×4 single-ended S-matrix converts to mixed-mode parameters with `scikit-rf`'s `se2gmm`. For a 4-port it assumes ports (0, 1) are the left-hand pair and (2, 3) the right-hand pair, and returns port 0 = differential left, 1 = differential right, 2 = common left, 3 = common right — hence the `o1`/`o2`/`o3`/`o4` ordering chosen in the geometry cell. Reference impedances become $2 z_0$ differential and $z_0/2$ common.

$Z_c$ and $\gamma$ then come from the ABCD matrix of each mode's 2-port. For a uniform, symmetric, reciprocal line

$$\mathrm{ABCD} = \begin{bmatrix} \cosh \gamma \ell & Z_c \sinh \gamma \ell \\ \sinh \gamma \ell / Z_c & \cosh \gamma \ell \end{bmatrix}$$

so $Z_c = \sqrt{B/C}$, and fixing that square root on the passive ($\mathrm{Re} > 0$) branch makes $\sinh \gamma\ell = B / Z_c$ unambiguous too. Then $e^{\gamma \ell} = \cosh \gamma\ell + \sinh \gamma\ell$ is fully determined and $\gamma \ell$ follows from its complex logarithm, with only the $2\pi$ of the phase left to unwrap over the sweep.

This differs from the eigenvector/eigenvalue approach used in the single-ended notebook, and deliberately so. Selecting the forward ABCD eigenvalue is only unambiguous while $|\beta \ell| < \pi$. Beyond that the two eigenvalues $e^{\pm j \beta \ell}$ of a low-loss line do not cross but **collide** at $\beta\ell = \pi$, so neither picking the most-negative phase nor tracking by continuity can stay on the forward wave — both silently return a wrong $n_{\mathrm{eff}}$. The T-bar section below pushes $n_{\mathrm{eff}}$ high enough for an 800 µm line to cross that threshold within this sweep. On two synthetic uncoupled 50 Ω lines with $n_{\mathrm{eff}} = 2$, the formulation used here recovers $Z_c$ = 50/100/25 Ω and $n_{\mathrm{eff}} = 2$ to machine precision out past $\beta\ell = 2\pi$, where the eigenvalue methods return 1.73.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import skrf as rf
from scipy.constants import speed_of_light
from skrf.calibration.deembedding import IEEEP370_SE_NZC_2xThru


def _positive_real_branch(z):
    return np.where(np.real(z) < 0, -z, z)


def extract_modal_parameters(net, length_m):
    """Return Zc, gamma and neff of a uniform symmetric 2-port from its ABCD.

    Branch-free: Zc = sqrt(B/C) on the passive branch determines
    sinh(gamma*l) = B/Zc, hence exp(gamma*l) = cosh + sinh, leaving only the
    2*pi of the complex log to unwrap. No eigenvalue selection, so the
    beta*l = pi eigenvalue collision of a low-loss line never arises.
    """
    a = net.a
    A, B, C, D = a[:, 0, 0], a[:, 0, 1], a[:, 1, 0], a[:, 1, 1]

    zc = _positive_real_branch(np.sqrt(B / C))
    cosh_gl = (A + D) / 2
    sinh_gl = B / zc
    exp_gl = cosh_gl + sinh_gl

    gamma_l = np.log(np.abs(exp_gl)) + 1j * np.unwrap(np.angle(exp_gl))
    gamma = gamma_l / length_m

    neff = np.full(len(net), np.nan)
    nonzero = net.f != 0
    neff[nonzero] = (
        np.imag(gamma[nonzero]) * speed_of_light / (2 * np.pi * net.f[nonzero])
    )

    # Reciprocity/symmetry residual: a uniform line has A == D and AD - BC == 1.
    symmetry_error = np.max(np.abs(A - D) / np.maximum(np.abs(A), 1e-30))

    return {
        "zc": zc,
        "gamma": gamma,
        "neff": neff,
        "symmetry_error": symmetry_error,
    }


def to_mixed_mode(sparams):
    """Split a 4-port GSGSG result into single-ended, differential and common 2-ports."""
    net = sparams.to_skrf()
    net.frequency.unit = "GHz"

    # Port order comes from the results parser, not from the add_wave_port call
    # order, and se2gmm's pairing convention depends on it. A stale cached run
    # (pre-dating named ports) falls back to p1..pN instead of o1..o4; those
    # are already in add_wave_port call order, so there is nothing to reorder.
    names = list(sparams.port_names)
    if len(names) != len(PORT_NAMES):
        raise ValueError(
            f"Expected {len(PORT_NAMES)} ports, got {names} — cannot infer "
            "mixed-mode pairing."
        )
    if set(names) >= set(PORT_NAMES):
        if names != list(PORT_NAMES):
            net.renumber([names.index(n) for n in PORT_NAMES], range(len(PORT_NAMES)))
    elif not all(n.startswith("p") and n[1:].isdigit() for n in names):
        raise ValueError(
            f"Port names {names} are neither {list(PORT_NAMES)} nor the "
            "numeric fallback (p1, p2, ...) — cannot infer mixed-mode pairing."
        )

    net_mm = net.copy()
    net_mm.se2gmm(p=2)

    z0 = np.real(net.z0[0, 0])
    assert np.allclose(np.real(net_mm.z0[0, :2]), 2 * z0), net_mm.z0[0]
    assert np.allclose(np.real(net_mm.z0[0, 2:]), z0 / 2), net_mm.z0[0]

    return {
        "single": net.subnetwork([0, 2]),  # upper line, left -> right
        "diff": net_mm.subnetwork([0, 1]),
        "comm": net_mm.subnetwork([2, 3]),
        "se": net,
        "mm": net_mm,
    }


def p370_extract(short_2xthru, long_fix_dut_fix, delta_length_m):
    """Extract the additional DUT length using a mirrored IEEE P370 split."""
    reference_z0 = float(np.real(np.median(short_2xthru.z0[:, 0])))
    deembedding = IEEEP370_SE_NZC_2xThru(
        dummy_2xthru=short_2xthru,
        z0=reference_z0,
        use_z_instead_ifft=True,
        name="100 um non-zero-length 2xThru",
    )
    dut = deembedding.deembed(long_fix_dut_fix)

    # Explicitly reconstruct both topologies to check fixture orientation.
    reconstructed_short = deembedding.s_side1 ** deembedding.s_side2.flipped()
    reconstructed_long = deembedding.s_side1**dut ** deembedding.s_side2.flipped()
    result = extract_modal_parameters(dut, delta_length_m)
    result.update(
        {
            "dut": dut,
            "deembedding": deembedding,
            "split_error": np.max(np.abs(reconstructed_short.s - short_2xthru.s)),
            "reembed_error": np.max(np.abs(reconstructed_long.s - long_fix_dut_fix.s)),
        }
    )
    return result

In [ ]:
nets = to_mixed_mode(results)

LENGTH_M = 800e-6
modes = {
    k: extract_modal_parameters(nets[k], LENGTH_M) for k in ("single", "diff", "comm")
}

for label, m in modes.items():
    print(
        f"{label:7s} Zc = {np.median(m['zc'].real):7.2f} ohm   "
        f"neff = {np.nanmedian(m['neff']):6.3f}   "
        f"A-D asymmetry = {m['symmetry_error']:.2e}"
    )

beta_l = 2 * np.pi * nets["diff"].f * np.nanmedian(modes["diff"]["neff"]) * LENGTH_M
print(f"\nmax |beta*l| over the sweep = {beta_l.max() / speed_of_light:.2f} rad")

In [ ]:
rf.stylely()

fig, ax = plt.subplots()
for label, style in (("single", "-"), ("diff", "--"), ("comm", ":")):
    ax.plot(
        nets[label].frequency.f_scaled,
        modes[label]["zc"].real,
        style,
        label=f"{label} mode",
    )
ax.set_xlabel(f"Frequency [{nets['diff'].frequency.unit}]")
ax.set_ylabel(r"$Re(Z_c)$ [Ohm]")
ax.set_title("GSGSG characteristic impedance")
ax.legend()

In [ ]:
fig, ax = plt.subplots()
for label, style in (("single", "-"), ("diff", "--"), ("comm", ":")):
    ax.plot(
        nets[label].frequency.f_scaled,
        modes[label]["neff"].real,
        style,
        label=f"{label} mode",
    )
ax.set_xlabel(f"Frequency [{nets['diff'].frequency.unit}]")
ax.set_ylabel(r"$n_{eff}$")
ax.set_title("GSGSG effective index")
ax.legend()

### How decoupled are the two lines?

At an 80 µm pitch the two lines are decoupled *by construction*, which means the mixed-mode transform is close to a pure rescaling: an ideal pair of independent lines has exactly $Z_{c,\mathrm{diff}} = 2 Z_{c,\mathrm{single}}$, $Z_{c,\mathrm{comm}} = Z_{c,\mathrm{single}}/2$ and $n_{\mathrm{eff,diff}} = n_{\mathrm{eff,comm}}$. The informative quantity is therefore not the mixed-mode values themselves but the **departure** from those identities, which measures the residual coupling directly:

- $Z_{c,\mathrm{diff}} / (2 Z_{c,\mathrm{single}}) - 1$ — deviates below zero as the lines couple
- $n_{\mathrm{eff,diff}} - n_{\mathrm{eff,comm}}$ — zero only for truly independent lines
- $|S_{dc}|$ — differential-to-common **mode conversion**
- $|S_{21}|$ between the two left-hand ports — direct near-end crosstalk

The mode-conversion term carries extra weight because the analysis above slices the mixed-mode matrix as `subnetwork([0, 1])` and `subnetwork([2, 3])`, which treats the differential and common modes as two independent 2-ports. That is only legitimate while $S_{dc} \approx S_{cd} \approx 0$; the symmetry of this structure should make it so, but it is an assumption worth confirming rather than presuming. It matters most in the T-bar section, where the shared central ground genuinely couples the two lines.

In [ ]:
fig, axes = plt.subplots(nrows=4, sharex=True, figsize=(8, 10))
freq_scaled = nets["diff"].frequency.f_scaled
unit = nets["diff"].frequency.unit

zc_ratio = modes["diff"]["zc"].real / (2 * modes["single"]["zc"].real) - 1
axes[0].plot(freq_scaled, 100 * zc_ratio)
axes[0].set_ylabel(r"$Z_{c,diff} / 2Z_{c,single} - 1$ [%]")
axes[0].axhline(0, color="k", lw=0.5)

axes[1].plot(freq_scaled, modes["diff"]["neff"].real - modes["comm"]["neff"].real)
axes[1].set_ylabel(r"$n_{eff,diff} - n_{eff,comm}$")
axes[1].axhline(0, color="k", lw=0.5)

# Mode conversion. Slicing the mixed-mode matrix into independent differential
# and common 2-ports is only valid while these stay small.
s_mm = nets["mm"].s
mode_conversion = np.maximum(np.abs(s_mm[:, 0, 2]), np.abs(s_mm[:, 2, 0]))
axes[2].plot(freq_scaled, 20 * np.log10(np.maximum(mode_conversion, 1e-20)))
axes[2].set_ylabel(r"$|S_{dc}|$ [dB]")

# Near-end crosstalk: port 2 (lower line, left) driven from port 1 (upper, left)
axes[3].plot(freq_scaled, 20 * np.log10(np.abs(nets["se"].s[:, 1, 0])))
axes[3].set_ylabel("near-end crosstalk [dB]")
axes[3].set_xlabel(f"Frequency [{unit}]")

axes[0].set_title("Residual coupling between the two lines")
plt.tight_layout()

print(f"max |Sdc| = {mode_conversion.max():.3e}")
print(
    f"max near-end crosstalk = {20 * np.log10(np.abs(nets['se'].s[:, 1, 0])).max():.1f} dB"
)

## Length verification

Three GSGSG electrodes (100, 400, 800 µm) are simulated under identical waveport and mesh settings. The 100 µm line is used as an <a href="https://scikit-rf.readthedocs.io/en/latest/api/calibration/generated/skrf.calibration.deembedding.IEEEP370_SE_NZC_2xThru.html">IEEE P370 NZC 2x-thru fixture</a> and de-embedded from the longer results to yield effective 300 µm (400−100) and 700 µm (800−100) DUT segments.

As in the single-ended notebook, the purpose is to see how much the port parasitics perturb the extracted parameters: the waveport boundary introduces a reactive discontinuity at each end whose magnitude is fixed regardless of line length, so the shortest line carries the largest fractional contamination and should deviate most. Here the de-embedding is applied **separately to the differential and common-mode 2-ports**, each of which is an ordinary 2-port that the P370 fixture handles directly.

In [ ]:
gsgsg100 = gsgsg_electrode(length=100)
gsgsg400 = gsgsg_electrode(length=400)
gsgsg800 = gsgsg_electrode(length=800)

In [ ]:
raw_results = []
for lc in [gsgsg100, gsgsg400, gsgsg800]:
    sim = setup_sim(lc)
    sim.mesh(preset="default", refined_mesh_size=2.0, max_mesh_size=40.0, fmax=60e9)
    raw_results.append(sim.run(wait=False, check_cache=True))

In [ ]:
import gsim

# Poll all jobs concurrently, download and parse results
raw_results = gsim.wait_for_results(raw_results)

In [ ]:
mm = [to_mixed_mode(r) for r in raw_results]

# De-embed the 100 um fixture from the 400 and 800 um lines, per mode
deembedded = {}
for mode in ("single", "diff", "comm"):
    deembedded[mode] = {
        300: p370_extract(mm[0][mode], mm[1][mode], delta_length_m=300e-6),
        700: p370_extract(mm[0][mode], mm[2][mode], delta_length_m=700e-6),
    }

for mode, lengths in deembedded.items():
    for length_um, res in lengths.items():
        print(
            f"{mode:7s} {length_um:3d} um   split_error={res['split_error']:.2e}   "
            f"reembed_error={res['reembed_error']:.2e}"
        )

In [ ]:
direct = {
    mode: {
        100: extract_modal_parameters(mm[0][mode], 100e-6),
        400: extract_modal_parameters(mm[1][mode], 400e-6),
        800: extract_modal_parameters(mm[2][mode], 800e-6),
    }
    for mode in ("single", "diff", "comm")
}

In [ ]:
fig, axes = plt.subplots(ncols=3, sharex=True, figsize=(13, 4))

for ax, mode in zip(axes, ("single", "diff", "comm"), strict=True):
    freq_scaled = mm[0][mode].frequency.f_scaled
    for length_um in (100, 400, 800):
        ax.plot(
            freq_scaled, direct[mode][length_um]["zc"].real, label=f"{length_um} um"
        )
    for length_um in (300, 700):
        ax.plot(
            freq_scaled,
            deembedded[mode][length_um]["zc"].real,
            "--",
            label=f"{length_um} um effective",
        )
    ax.set_title(f"{mode} mode")
    ax.set_xlabel(f"Frequency [{mm[0][mode].frequency.unit}]")

axes[0].set_ylabel(r"$Re(Z_c)$ [Ohm]")
axes[0].legend(fontsize="small")
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(ncols=3, sharex=True, figsize=(13, 4))

for ax, mode in zip(axes, ("single", "diff", "comm"), strict=True):
    freq_scaled = mm[0][mode].frequency.f_scaled
    for length_um in (100, 400, 800):
        ax.plot(
            freq_scaled, direct[mode][length_um]["neff"].real, label=f"{length_um} um"
        )
    for length_um in (300, 700):
        ax.plot(
            freq_scaled,
            deembedded[mode][length_um]["neff"].real,
            "--",
            label=f"{length_um} um effective",
        )
    ax.set_title(f"{mode} mode")
    ax.set_xlabel(f"Frequency [{mm[0][mode].frequency.unit}]")

axes[0].set_ylabel(r"$n_{eff}$")
axes[0].legend(fontsize="small")
plt.tight_layout()

### Distributed RLGC of the differential mode

In [ ]:
def rlgc(net, params):
    """Distributed R, L, G, C per metre from gamma and Zc."""
    omega = net.frequency.w
    series_per_m = params["gamma"] * params["zc"]
    shunt_per_m = params["gamma"] / params["zc"]
    return {
        "R": np.real(series_per_m),
        "L": np.imag(series_per_m) / omega * 1e6,  # uH/m
        "G": np.real(shunt_per_m),
        "C": np.imag(shunt_per_m) / omega * 1e12,  # pF/m
    }


def plot_rlgc(entries, title):
    """entries: list of (label, net, params, plot kwargs)."""
    fig, axes = plt.subplots(nrows=2, ncols=2, sharex=True, figsize=(9, 5))
    axes = axes.ravel()
    labels = (
        (r"$R \quad [\Omega / m]$", "R"),
        (r"$L \quad [\mu H/m]$", "L"),
        (r"$G \quad [S/m]$", "G"),
        (r"$C \quad [pF / m]$", "C"),
    )
    for label, net, params, kwargs in entries:
        values = rlgc(net, params)
        for ax, (ylabel, key) in zip(axes, labels, strict=True):
            ax.plot(net.frequency.f_scaled, values[key], label=label, **kwargs)
            ax.set_ylabel(ylabel)

    unit = entries[0][1].frequency.unit
    axes[2].set_xlabel(f"Frequency [{unit}]")
    axes[3].set_xlabel(f"Frequency [{unit}]")
    axes[0].legend(fontsize="small")
    fig.suptitle(title)
    plt.tight_layout()
    return fig

In [ ]:
plot_rlgc(
    [
        (
            f"{mode} (700 um effective)",
            deembedded[mode][700]["dut"],
            deembedded[mode][700],
            {},
        )
        for mode in ("single", "diff", "comm")
    ],
    "GSGSG distributed parameters",
)

## T-bar loaded electrode (MZM travelling-wave electrode)

A Mach-Zehnder modulator does not drive a bare transmission line. The modulator segments hang off the electrode as a periodic capacitive load, usually reached through **T-bars**: a stem out of the signal electrode ending in a cross arm, facing a mirrored T-bar from the ground, with the modulator capacitance across the remaining gap. This is a capacitively-loaded travelling-wave electrode (CL-TWE). The added shunt capacitance per unit length lowers $Z_c$ and raises $n_{\mathrm{eff}}$ — which is the point, since matching $n_{\mathrm{eff}}$ to the optical group index is what sets the modulator's bandwidth.

Here a **pair** of facing T-bars sits in each signal-to-central-ground gap, in every period, on both halves of the structure. Two parameters are constrained rather than free:

- **`tbar_period` must divide every simulated length.** The P370 NZC fixture assumes a uniform line, so a fractional number of periods invalidates the de-embedding. For lengths of 100/400/800 µm that allows {10, 20, 25, 50} µm; 25 µm is used here, giving 4/16/32 periods and 12/28 periods for the de-embedded segments.
- **`tbar_gap` must stay at or above the mesh refinement size**, or the loading capacitance the section is about goes unresolved. At `refined_mesh_size=2.0` a 3 µm gap with 2 µm arms was found to mesh with a minimum element quality of 0.11, essentially matching the unloaded line's 0.13; a 2 µm gap or a 3 µm arm dropped it to 0.003–0.04, and *finer* refinement made it worse rather than better. Check `sim._last_mesh_result.mesh_stats["quality"]` if these are changed.

In [ ]:
@gf.cell
def gsgsg_tbar_electrode(
    length: float = 800,
    s_width: float = 20,
    g_width: float = 40,
    gap_width: float = 15,
    signal_pitch: float = 80,
    tbar_period: float = 25.0,
    tbar_stem_width: float = 4.0,
    tbar_arm_length: float = 10.0,
    tbar_arm_width: float = 2.0,
    tbar_gap: float = 3.0,
    layer=LAYER.TopMetal2drawing,
) -> gf.Component:
    """
    GSGSG electrode capacitively loaded with T-bars (MZM CL-TWE).

    In every period, each signal-to-central-ground gap carries a pair of facing
    T-bars: one grown from the signal electrode, one from the central ground,
    with their cross arms separated by `tbar_gap`.

    Args:
        length: horizontal length of the electrodes
        s_width: width of each signal electrode
        g_width: width of the two outer ground electrodes
        gap_width: gap between a signal electrode and its adjacent grounds
        signal_pitch: centre-to-centre spacing of the two signal electrodes
        tbar_period: longitudinal pitch of the T-bar pairs. Must divide every
            simulated length or the IEEE P370 de-embedding is invalid.
        tbar_stem_width: width of the T stem, along x
        tbar_arm_length: length of the T cross arm, along x
        tbar_arm_width: thickness of the T cross arm, along y
        tbar_gap: gap between the two facing cross arms. Keep at or above the
            refined mesh size or the loading capacitance is unresolved.
        layer: layer for the metal
    """
    c = gf.Component()
    base = c << gsgsg_electrode(
        length=length,
        s_width=s_width,
        g_width=g_width,
        gap_width=gap_width,
        signal_pitch=signal_pitch,
        layer=layer,
    )

    # Each T occupies (gap_width - tbar_gap) / 2 of the gap: stem, then arm.
    stem_length = (gap_width - tbar_gap) / 2 - tbar_arm_width
    if stem_length <= 0:
        raise ValueError(
            f"tbar_gap={tbar_gap} and tbar_arm_width={tbar_arm_width} leave no "
            f"room for a stem inside gap_width={gap_width}."
        )

    n_periods = round(length / tbar_period)
    if abs(n_periods * tbar_period - length) > 1e-9:
        raise ValueError(
            f"tbar_period={tbar_period} does not divide length={length}; a "
            "fractional number of periods breaks the P370 de-embedding."
        )

    s_inner = signal_pitch / 2 - s_width / 2  # signal edge facing the centre
    g_inner = s_inner - gap_width  # central ground edge

    x_centers = [-length / 2 + (i + 0.5) * tbar_period for i in range(n_periods)]

    for sign in (+1, -1):
        for x0 in x_centers:
            # T grown inward from the signal electrode, and its mirror grown
            # outward from the central ground.
            for y_root, direction in ((s_inner, -1), (g_inner, +1)):
                stem = c << gf.c.rectangle(
                    (tbar_stem_width, stem_length), centered=True, layer=layer
                )
                stem.move((x0, sign * (y_root + direction * stem_length / 2)))

                arm = c << gf.c.rectangle(
                    (tbar_arm_length, tbar_arm_width), centered=True, layer=layer
                )
                arm.move(
                    (
                        x0,
                        sign
                        * (y_root + direction * (stem_length + tbar_arm_width / 2)),
                    )
                )

    c.add_ports(base.ports)
    c.info["s_width"] = s_width
    c.info["g_width"] = g_width
    c.info["gap_width"] = gap_width
    c.info["signal_pitch"] = signal_pitch
    c.info["tbar_period"] = tbar_period
    c.info["tbar_gap"] = tbar_gap
    c.info["n_periods"] = n_periods
    return c


ct = gsgsg_tbar_electrode(length=100)
print(dict(ct.info))
ct

In [ ]:
tbar100 = gsgsg_tbar_electrode(length=100)
tbar400 = gsgsg_tbar_electrode(length=400)
tbar800 = gsgsg_tbar_electrode(length=800)

tbar_raw = []
for lc in [tbar100, tbar400, tbar800]:
    sim = setup_sim(lc, output_dir="./palace-sim-gsgsg-tbar")
    sim.mesh(preset="default", refined_mesh_size=2.0, max_mesh_size=40.0, fmax=60e9)
    stats = sim._last_mesh_result.mesh_stats
    print(
        f"length={lc.info['n_periods'] * lc.info['tbar_period']:5.0f} um  "
        f"nodes={stats['nodes']:6d}  min quality={stats['quality']['min']:.3f}  "
        f"invalid={stats['sicn']['invalid']}"
    )
    tbar_raw.append(sim.run(wait=False, check_cache=True))

In [ ]:
tbar_raw = gsim.wait_for_results(tbar_raw)

In [ ]:
tbar_mm = [to_mixed_mode(r) for r in tbar_raw]

tbar_deembedded = {}
for mode in ("single", "diff", "comm"):
    tbar_deembedded[mode] = {
        300: p370_extract(tbar_mm[0][mode], tbar_mm[1][mode], delta_length_m=300e-6),
        700: p370_extract(tbar_mm[0][mode], tbar_mm[2][mode], delta_length_m=700e-6),
    }

for mode in ("single", "diff", "comm"):
    unloaded = deembedded[mode][700]
    loaded = tbar_deembedded[mode][700]
    print(
        f"{mode:7s} Zc {np.median(unloaded['zc'].real):6.2f} -> "
        f"{np.median(loaded['zc'].real):6.2f} ohm    "
        f"neff {np.nanmedian(unloaded['neff']):5.3f} -> "
        f"{np.nanmedian(loaded['neff']):5.3f}"
    )

### Loaded vs unloaded

Both extractions use the 700 µm de-embedded segment, so the port parasitics are removed from each and the comparison isolates the effect of the T-bars.

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=3, sharex=True, figsize=(13, 7))

for col, mode in enumerate(("single", "diff", "comm")):
    unloaded = deembedded[mode][700]
    loaded = tbar_deembedded[mode][700]
    freq_scaled = unloaded["dut"].frequency.f_scaled

    axes[0, col].plot(freq_scaled, unloaded["zc"].real, label="unloaded")
    axes[0, col].plot(freq_scaled, loaded["zc"].real, "--", label="T-bar loaded")
    axes[0, col].set_title(f"{mode} mode")

    axes[1, col].plot(freq_scaled, unloaded["neff"].real, label="unloaded")
    axes[1, col].plot(freq_scaled, loaded["neff"].real, "--", label="T-bar loaded")
    axes[1, col].set_xlabel(f"Frequency [{unloaded['dut'].frequency.unit}]")

axes[0, 0].set_ylabel(r"$Re(Z_c)$ [Ohm]")
axes[1, 0].set_ylabel(r"$n_{eff}$")
axes[0, 0].legend(fontsize="small")
fig.suptitle("Effect of T-bar capacitive loading (700 um de-embedded segment)")
plt.tight_layout()

In [ ]:
plot_rlgc(
    [
        ("unloaded", deembedded["diff"][700]["dut"], deembedded["diff"][700], {}),
        (
            "T-bar loaded",
            tbar_deembedded["diff"][700]["dut"],
            tbar_deembedded["diff"][700],
            {"linestyle": "--"},
        ),
    ],
    "Differential mode: distributed parameters, unloaded vs T-bar loaded",
)

The loading shows up in the distributed parameters as an increase in $C$ with $L$ essentially unchanged, since the T-bars add shunt capacitance without altering the current path. That is the CL-TWE design lever: $Z_c = \sqrt{L/C}$ falls and $n_{\mathrm{eff}} \propto \sqrt{LC}$ rises, so period, arm length and gap can be traded off to hit both a 50 Ω match and velocity matching to the optical mode.

Note also that the differential and common modes no longer track each other the way they did on the bare line: the T-bars tie both signal electrodes to the same central ground, so the shared return path couples the two lines even though their 80 µm pitch leaves them decoupled when unloaded. The residual-coupling metrics plotted earlier are worth re-running on the loaded structure for that reason.